In [1]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.6-flash",  # Switched to an alternative Flash model
    contents="Give me a simple Indian dal recipe (in less than 10 lines) with its calorie count.",
)

print(response.text)

**Simple Everyday Yellow Dal Tadka (4 Servings)**

**Ingredients:** 1 cup Toor/Moong dal, 3 cups water, ½ tsp turmeric, salt to taste, 1 tbsp ghee, 1 tsp cumin seeds, 2 minced garlic cloves, 1 chopped green chili.

1. Wash dal and pressure cook with water, turmeric, and salt for 3-4 whistles (until soft).
2. Mash the cooked dal slightly with a whisk to smooth it out.
3. Heat ghee in a small pan; add cumin seeds, garlic, and chili, frying for 1 minute until golden.
4. Pour this hot ghee tempering (tadka) directly into the dal and mix well.
5. Garnish with fresh coriander leaves and serve warm with rice.

**Calories:** ~195 kcal per serving (1 bowl)


In [2]:
import json

response = client.models.generate_content(
    model="gemini-3.6-flash",  # Switched to an alternative Flash model
    contents='''Give me a simple Indian dal recipe,
    Return only JSON with these keys: title, calories, prep_time_minutes.'''
)

print(response.text)

```json
{
  "title": "Simple Yellow Tadka Dal",
  "calories": 220,
  "prep_time_minutes": 10
}
```


In [4]:
json.loads('''
{
  "title": "Simple Yellow Tadka Dal",
  "calories": 220,
  "prep_time_minutes": 10
}
''')

{'title': 'Simple Yellow Tadka Dal', 'calories': 220, 'prep_time_minutes': 10}

In [5]:
# Convert the string into dictionary
json.loads(response.text) # gives JSONDecodeError because of extra preamble

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [10]:
# Fix the ```json fences off before parsing
clean = response.text.strip()
clean = clean.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

data = json.loads(clean)
print("Calories:", data["calories"])
print(type(data))

Calories: 220
<class 'dict'>


##### We need a better and more robust way. And for that, we use Pydantic classes.

In [11]:
from pydantic import BaseModel
from google.genai import types

class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    calories: int
    prep_time_minutes: int

response = client.models.generate_content(
    model="gemini-3.6-flash",  
    contents="Give me a simple Indian dal recipe,",
    config=types.GenerateContentConfig(
        response_mime_type= "application/json", # reply in JSON
        response_schema=Recipe  # "...in this shape"
    ),
)

print(response.text)

{"title":"Simple Yellow Tadka Dal","ingredients":["1 cup yellow moong dal","3 cups water","1/2 tsp turmeric powder","1 tsp salt","2 tbsp ghee or oil","1 tsp cumin seeds","1 green chili, finely chopped","1 small onion, finely chopped","2 cloves garlic, minced","1 tbsp fresh coriander, chopped"],"calories":220,"prep_time_minutes":25}


In [12]:
response.parsed

Recipe(title='Simple Yellow Tadka Dal', ingredients=['1 cup yellow moong dal', '3 cups water', '1/2 tsp turmeric powder', '1 tsp salt', '2 tbsp ghee or oil', '1 tsp cumin seeds', '1 green chili, finely chopped', '1 small onion, finely chopped', '2 cloves garlic, minced', '1 tbsp fresh coriander, chopped'], calories=220, prep_time_minutes=25)

In [13]:
recipe = response.parsed

In [14]:
recipe.calories

220

In [ ]:
response = requests.get(
    